# Non-small-cell lung cancer · GSE292299

This spatial transcriptomics notebook uses the dataset-specific marker panel in the cells below. Prepare the inputs listed in [the data guide](../../data/README.md). Start with a fresh kernel and run the annotation cells in order. The last section contains optional comparison methods. The notebook writes its files under `results/runs/spatial/lung_gse292299/`.


In [ ]:
from pathlib import Path
import os

# Both a kernel started alongside the notebook and one started at the repository root work.
start = Path.cwd().resolve()
ROOT = next((p for p in (start, *start.parents) if (p / "data" / "datasets.tsv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from a checkout containing data/datasets.tsv.")
RAW = ROOT / "data" / "raw"
PREPARED = ROOT / "data" / "prepared"
RUN_DIR = ROOT / "results" / "runs" / "spatial" / "lung_gse292299"
RUN_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(RUN_DIR)
RUN_COMPARISONS = False     # Enable after installing the optional comparison tools.
RESTORE_LUNG_LABELS = False


In [ ]:
# ============================================================
# GSE292299_SELECTED
# NSCLC Lung Cancer Spatial Transcriptomics
#
# Load selected Visium samples and merge into one AnnData
# ============================================================

from pathlib import Path
import shutil
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. Main selected dataset folder
# ------------------------------------------------------------

data_dir = RAW / "GSE292299_SELECTED"


# ------------------------------------------------------------
# 2. Selected NSCLC samples
#
# Based on the subset we downloaded earlier
# ------------------------------------------------------------

sample_info = {

    "P1": {
        "gsm": "GSM8855703",
        "condition": "NSCLC"
    },

    "P2": {
        "gsm": "GSM8855704",
        "condition": "NSCLC"
    },

    "P4": {
        "gsm": "GSM8855706",
        "condition": "NSCLC"
    },

    "P5": {
        "gsm": "GSM8855707",
        "condition": "NSCLC"
    },
}

samples = list(sample_info.keys())

print("Selected lung samples:")
print(samples)

In [ ]:
# ============================================================
# 3. Helper functions
# ============================================================

def find_single_file(root, pattern):
    """
    Recursively find one file matching pattern.
    """
    matches = list(
        Path(root).rglob(pattern)
    )

    if len(matches) == 0:
        return None

    if len(matches) > 1:
        print(
            f"WARNING: multiple matches for {pattern}:"
        )
        for x in matches:
            print("   ", x)

    return matches[0]


def find_spatial_directory(root):
    """
    Find directory containing tissue_positions_list.csv
    or tissue_positions.csv.
    """

    root = Path(root)

    position_files = list(
        root.rglob("tissue_positions_list.csv")
    )

    position_files += list(
        root.rglob("tissue_positions.csv")
    )

    if len(position_files) == 0:
        return None

    return position_files[0].parent

In [ ]:
# ============================================================
# 4. Locate H5 and spatial data
# ============================================================

discovered = {}

for sample_name, info in sample_info.items():

    gsm = info["gsm"]

    print("\n" + "=" * 60)
    print(sample_name, gsm)
    print("=" * 60)

    # Search within the selected dataset folder
    h5_file = find_single_file(
        data_dir,
        f"{gsm}_*_filtered_feature_bc_matrix.h5"
    )

    spatial_dir = None

    # Search folders associated with this GSM
    candidate_dirs = [
        p
        for p in data_dir.rglob("*")
        if p.is_dir()
        and gsm in str(p)
    ]

    for candidate in candidate_dirs:

        tmp = find_spatial_directory(
            candidate
        )

        if tmp is not None:
            spatial_dir = tmp
            break

    print("H5:")
    print(h5_file)

    print("Spatial directory:")
    print(spatial_dir)

    if h5_file is None:
        raise FileNotFoundError(
            f"H5 file not found for {sample_name} ({gsm})"
        )

    if spatial_dir is None:
        raise FileNotFoundError(
            f"Spatial folder not found for {sample_name} ({gsm})"
        )

    discovered[sample_name] = {
        "gsm": gsm,
        "h5": h5_file,
        "spatial": spatial_dir
    }

In [ ]:
# ============================================================
# 5. Prepare Scanpy-compatible Visium folders
# ============================================================

prepared_root = (
    data_dir /
    "prepared_visium"
)

prepared_root.mkdir(
    parents=True,
    exist_ok=True
)

sample_paths = {}

for sample_name, info in discovered.items():

    gsm = info["gsm"]

    sample_folder = (
        prepared_root /
        sample_name
    )

    spatial_dest = (
        sample_folder /
        "spatial"
    )

    sample_folder.mkdir(
        parents=True,
        exist_ok=True
    )

    spatial_dest.mkdir(
        parents=True,
        exist_ok=True
    )

    # --------------------------------------------------------
    # Copy H5 with standard name
    # --------------------------------------------------------

    h5_dest = (
        sample_folder /
        "filtered_feature_bc_matrix.h5"
    )

    shutil.copy2(
        info["h5"],
        h5_dest
    )

    # --------------------------------------------------------
    # Copy spatial files
    # --------------------------------------------------------

    for f in Path(
        info["spatial"]
    ).iterdir():

        if f.is_file():

            shutil.copy2(
                f,
                spatial_dest / f.name
            )

    sample_paths[sample_name] = str(
        sample_folder
    )

    print(
        sample_name,
        "prepared at:",
        sample_folder
    )

In [ ]:
# ============================================================
# 6. Inspect prepared folders
# ============================================================

for sample_name, path in sample_paths.items():

    print(
        "\n---",
        sample_name,
        "---"
    )

    for f in Path(path).rglob("*"):

        if f.is_file():

            print(
                f.relative_to(path)
            )

In [ ]:
# ============================================================
# 7. Load selected NSCLC samples
# ============================================================

adatas = []

for sample_name, path in sample_paths.items():

    print("\nLoading:", sample_name)

    adata_sample = sc.read_visium(
        path=path,
        count_file="filtered_feature_bc_matrix.h5"
    )

    adata_sample.var_names_make_unique()

    gsm = sample_info[
        sample_name
    ]["gsm"]

    # --------------------------------------------------------
    # Metadata
    # --------------------------------------------------------

    adata_sample.obs[
        "sample"
    ] = sample_name

    adata_sample.obs[
        "gsm"
    ] = gsm

    adata_sample.obs[
        "sample_id"
    ] = (
        f"{gsm}_{sample_name}"
    )

    adata_sample.obs[
        "condition"
    ] = "NSCLC"

    adata_sample.obs[
        "dataset"
    ] = "GSE292299"

    adata_sample.obs[
        "cancer_type"
    ] = "Non_small_cell_lung_cancer"

    # --------------------------------------------------------
    # Preserve raw barcode
    # --------------------------------------------------------

    adata_sample.obs[
        "barcode"
    ] = (
        adata_sample.obs_names
        .astype(str)
    )

    # --------------------------------------------------------
    # Unique observation names
    #
    # P1_AAAC...
    # P2_AAAC...
    # --------------------------------------------------------

    adata_sample.obs_names = pd.Index(
        [
            f"{sample_name}_{bc}"
            for bc in adata_sample.obs[
                "barcode"
            ].astype(str)
        ]
    )

    # --------------------------------------------------------
    # Keep tissue spots only
    # --------------------------------------------------------

    if (
        "in_tissue"
        in adata_sample.obs.columns
    ):

        adata_sample = adata_sample[
            adata_sample.obs[
                "in_tissue"
            ].astype(int)
            == 1
        ].copy()

    adatas.append(
        adata_sample
    )

    print(
        sample_name,
        adata_sample.shape
    )

In [ ]:
# ============================================================
# 8. Merge selected lung samples
# ============================================================

adata_spatial_lung = ad.concat(
    adatas,
    join="outer",
    merge="same",
    label="batch",
    keys=samples,
    index_unique=None
)

print(
    "\nCombined lung-cancer object:"
)

print(
    adata_spatial_lung
)

In [ ]:
# Preprocess spatial data for scAICME-style semi-supervised clustering
import scanpy as sc
import numpy as np
import pandas as pd


# keep only tissue spots if available
if "in_tissue" in adata_spatial_lung.obs.columns:
    adata_spatial_lung= adata_spatial_lung[adata_spatial_lung.obs["in_tissue"].astype(int) == 1].copy()

adata=adata_spatial_lung
# Keep a raw copy before HVG restriction if needed later
print(f"START  | cells={adata.n_obs:,}  genes={adata.n_vars:,}")
adata.var_names = adata.var_names.astype(str)
adata.var_names_make_unique()
vn_up = adata.var_names.str.upper()

# --- Define QC gene sets and SAVE as columns in adata.var ---
# Human and mouse 'mt-' both become 'MT-' after uppercasing
adata.var["mt"]   = vn_up.str.startswith("MT-")
adata.var["ribo"] = vn_up.str.startswith(("RPS","RPL","MRPS","MRPL"))
adata.var["hb"]   = adata.var_names.str.match(r"^(HB[ABEDM][A-Z0-9]*)", case=False)

# --- Compute QC metrics using those var columns ---
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt","ribo","hb"],   # <- names of columns in adata.var
    percent_top=None,
    log1p=False,
    inplace=True
)

print(adata.obs[["n_genes_by_counts","total_counts","pct_counts_mt"]].describe())

# --- Choose thresholds (data-driven) ---
min_genes = 200
min_cells_gene = 3
umi_hi   = np.percentile(adata.obs["total_counts"], 99.5)
genes_hi = np.percentile(adata.obs["n_genes_by_counts"], 99.5)
mito_hi  = max(20.0, np.percentile(adata.obs["pct_counts_mt"], 95))

print(f"Thresholds → min_genes={min_genes}, umi_hi≈{umi_hi:.0f}, genes_hi≈{genes_hi:.0f}, mito_hi≈{mito_hi:.1f}%")

keep_cells = (
    (adata.obs["n_genes_by_counts"] >= min_genes) &
    (adata.obs["n_genes_by_counts"] <= genes_hi) &
    (adata.obs["total_counts"]      <= umi_hi) &
    (adata.obs["pct_counts_mt"]     <= mito_hi)
)
before = adata.n_obs
adata = adata[keep_cells].copy()
print(f"CELL FILTER | kept {adata.n_obs:,}/{before:,} ({adata.n_obs/before:.1%})")

# --- Filter low-prevalence genes ---
before_g = adata.n_vars
sc.pp.filter_genes(adata, min_cells=min_cells_gene)
print(f"GENE FILTER | kept {adata.n_vars:,}/{before_g:,} (≥{min_cells_gene} cells)")

# --- Normalize & log1p ---
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata
print(f"READY | cells={adata.n_obs:,} genes={adata.n_vars:,}")


In [ ]:
adata

In [ ]:
# Optional restoration of a previous run. A fresh run computes labels below.
if (RUN_DIR / 'scAICME_spatial_lung_labels.csv').is_file() and RESTORE_LUNG_LABELS:
    import pandas as pd

    # ============================================================
    # Load saved scAICME lung labels
    # ============================================================

    labels_df = pd.read_csv(
        "scAICME_spatial_lung_labels.csv"
    )

    print(labels_df.shape)
    print(labels_df.head())

    # ------------------------------------------------------------
    # The first column is the original obs_name
    # ------------------------------------------------------------

    labels_df = labels_df.rename(
        columns={
            "Unnamed: 0": "obs_name"
        }
    )

    # ------------------------------------------------------------
    # Make sure everything is string
    # ------------------------------------------------------------

    labels_df["obs_name"] = labels_df["obs_name"].astype(str)
    labels_df["sample"] = labels_df["sample"].astype(str)
    labels_df["barcode"] = labels_df["barcode"].astype(str)

    # ============================================================
    # Method 1: match directly by obs_names
    # ============================================================

    label_map = (
        labels_df
        .drop_duplicates("obs_name")
        .set_index("obs_name")[
            "label_consensus"
        ]
    )

    adata.obs["label_consensus"] = (
        adata.obs_names
        .astype(str)
        .map(label_map)
    )

    adata.obs["label_consensus"] = pd.Categorical(
        adata.obs["label_consensus"]
    )

    # ============================================================
    # Check
    # ============================================================

    print(
        "\nMatched labels:",
        adata.obs["label_consensus"].notna().sum(),
        "/",
        adata.n_obs
    )

    print(
        "\nLabel distribution:"
    )

    print(
        adata.obs["label_consensus"]
        .value_counts(dropna=False)
    )

    print(
        "\nSample-wise distribution:"
    )

    print(
        pd.crosstab(
            adata.obs["sample"],
            adata.obs["label_consensus"]
        )
    )

In [ ]:
PBMC_MARKERS = {

    "Tumor_Epithelial": [
        "EPCAM",
        "KRT8",
        "KRT18",
        "KRT19"
    ],

    "Fibroblast": [
        "COL1A1",
        "COL1A2",
        "DCN",
        "LUM"
    ],

    "Endothelial": [
        "PECAM1",
        "VWF",
        "CLDN5",
        "CDH5"
    ],

    "Tcell": [
        "CD3D",
        "CD3E",
        "TRAC"
    ],

    "NK": [
        "NKG7",
        "GNLY",
        "KLRD1",
        "PRF1"
    ],

    "Bcell": [
        "MS4A1",
        "CD79A",
        "CD74"
    ],

    "Plasma": [
        "MZB1",
        "JCHAIN",
        "SDC1"
    ],

    "Myeloid": [
        "LST1",
        "TYROBP",
        "FCER1G"
    ],

    "TAM_Macrophage": [
        "C1QA",
        "C1QB",
        "C1QC",
        "APOE",
        "CD68"
    ],

    "Neutrophil": [
        "FCGR3B",
        "CSF3R",
        "S100A8",
        "S100A9"
    ]
}

In [ ]:
# Method 1: adaptive marker-based weak seed labeling on spatial transcriptomics spots
import scanpy as sc
import numpy as np
import pandas as pd

def adaptive_marker_threshold(adata, gene, quantile=0.75):
    vals = adata[:, gene].X
    if hasattr(vals, "toarray"):
        vals = vals.toarray()
    vals = np.asarray(vals).ravel()
    vals = vals[vals > 0]
    if vals.size == 0:
        return np.inf
    return float(np.quantile(vals, quantile))

def weak_label_quota_with_min_cells(
    adata,
    marker_sets,
    gene_quantile=0.75,
    target_frac=0.50,
    min_cells_per_type=20,
    min_score=0.2,
    verbose=True
):
    N = adata.n_obs
    target_total = int(round(target_frac * N))
    scores = {}
    for cell_type, genes in marker_sets.items():
        genes = [g for g in genes if g in adata.var_names]
        if len(genes) == 0:
            continue
        thr = {g: adaptive_marker_threshold(adata, g, quantile=gene_quantile) for g in genes}
        M = sc.get.obs_df(adata, keys=list(thr.keys()))
        expr_high = M.gt(pd.Series(thr))
        frac_high = expr_high.sum(axis=1) / len(thr)
        scores[cell_type] = frac_high.astype(float)

    if len(scores) == 0:
        raise ValueError("No marker genes found in adata.var_names")

    score_df = pd.DataFrame(scores, index=adata.obs_names)
    best_score = score_df.max(axis=1).astype(float)
    best_type  = score_df.idxmax(axis=1).astype(str)

    eligible_mask = best_score >= float(min_score)
    eligible_types_counts = best_type[eligible_mask].value_counts()
    keep_types = eligible_types_counts[eligible_types_counts >= min_cells_per_type].index.tolist()

    if len(keep_types) == 0:
        adata.obs["weak_label"] = pd.Categorical(["unlabeled"] * N)
        adata.obs["weak_conf"]  = best_score.values
        return adata, score_df

    max_types_possible = max(1, target_total // max(1, min_cells_per_type))
    if len(keep_types) > max_types_possible:
        keep_types = eligible_types_counts.loc[keep_types].sort_values(ascending=False).index[:max_types_possible].tolist()

    base = {t: min_cells_per_type for t in keep_types}
    base_total = sum(base.values())
    remaining = max(0, target_total - base_total)
    avail = eligible_types_counts.loc[keep_types].to_dict()
    extra = {t: 0 for t in keep_types}

    if remaining > 0:
        weights = np.array([max(0, avail[t] - base[t]) for t in keep_types], dtype=float)
        if weights.sum() > 0:
            props = weights / weights.sum()
            raw_extra = np.floor(props * remaining).astype(int)
            for t, e in zip(keep_types, raw_extra):
                extra[t] = int(e)
            leftover = remaining - sum(extra.values())
            if leftover > 0:
                order = np.argsort(-props)
                k = 0
                while leftover > 0:
                    t = keep_types[order[k % len(order)]]
                    extra[t] += 1
                    leftover -= 1
                    k += 1

    quota = {t: min(base[t] + extra[t], avail[t]) for t in keep_types}

    weak_label = np.array(["unlabeled"] * N, dtype=object)
    best_type_arr = best_type.values
    best_score_arr = best_score.values
    eligible_idx = np.where(eligible_mask.values)[0]

    for t in keep_types:
        idx_t = eligible_idx[best_type_arr[eligible_idx] == t]
        idx_sorted = idx_t[np.argsort(-best_score_arr[idx_t])]
        pick = idx_sorted[:quota[t]]
        weak_label[pick] = t

    adata.obs["weak_label"] = pd.Categorical(weak_label)
    adata.obs["weak_conf"]  = best_score.values

    if verbose:
        print(adata.obs["weak_label"].value_counts())
        print(f"Labeled fraction: {(adata.obs['weak_label'] != 'unlabeled').mean():.2%}")

    return adata, score_df

adata, marker_score_df = weak_label_quota_with_min_cells(
    adata,
    PBMC_MARKERS,
    gene_quantile=0.75,
    target_frac=0.60,
    min_cells_per_type=10,
    min_score=0.20,
    verbose=True
)


#Method3

import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

def row_softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)
    E = np.exp(Z)
    return E / (E.sum(axis=1, keepdims=True) + 1e-12)

def gcn_seed_labeling_quota(
    adata,
    score_df,                         # cells x types marker_score_df
    target_frac=0.40,                 # desired final labeled fraction
    temperature=0.25,                 # 0.15–0.5 (lower = sharper)
    alpha=0.65,                       # 0.5–0.8 (lower = less oversmoothing)
    n_iter=50,
    tol=1e-4,                         # early stop
    # adaptive gates (quantiles)
    q_score=0.40,                     # gate on top1 >= quantile(top1)
    q_margin=0.35,                    # gate on margin >= quantile(margin)
    # quota controls
    min_cells_per_type=100,           # ensure each type has >= this many seeds (if possible)
    # minimum-size filter after quota (keep rare types!)
    min_cells_per_type_pct=0.001,     # 0.1% of n (≈68 for 68k)
    min_cells_floor=30,               # absolute min
    # graph construction
    neighbors_k=30,
    pca_comps=50,
    use_rep="X_pca",
    label_col="weak_label_gcn",
    conf_col="seed_conf_gcn",
    verbose=True
):
    # ---------- 0) Align score_df ----------
    if not isinstance(score_df, pd.DataFrame):
        raise ValueError("score_df must be a pandas DataFrame (cells x types).")

    score_df = score_df.copy()
    score_df.index = score_df.index.astype(str)
    score_df = score_df.reindex(adata.obs_names.astype(str))
    score_df = score_df.loc[:, score_df.notna().any(axis=0)]
    if score_df.shape[1] == 0:
        raise ValueError("score_df has no usable type columns after filtering.")

    types = score_df.columns.astype(str).tolist()
    S0 = score_df.fillna(0.0).to_numpy(dtype=float)

    # ---------- 1) Build sharpened probability seeds ----------
    # softmax produces probability-like vectors; temperature sharpens
    Y0 = row_softmax(S0 / float(temperature))

    # ---------- 2) Ensure PCA + neighbors ----------
    if (use_rep not in adata.obsm) or (adata.obsm[use_rep] is None):
        sc.pp.pca(adata, n_comps=pca_comps, svd_solver="arpack")
        use_rep = "X_pca"

    if "connectivities" not in adata.obsp:
        sc.pp.neighbors(
            adata,
            n_neighbors=neighbors_k,
            n_pcs=min(pca_comps, adata.obsm[use_rep].shape[1])
        )

    A = adata.obsp["connectivities"]
    if not sparse.isspmatrix(A):
        A = sparse.csr_matrix(A)

    N = adata.n_obs
    I = sparse.eye(N, format="csr")
    A_tilde = (A + I).tocsr()

    deg = np.asarray(A_tilde.sum(axis=1)).ravel()
    deg_inv_sqrt = 1.0 / np.sqrt(deg + 1e-12)
    D_inv_sqrt = sparse.diags(deg_inv_sqrt, format="csr")
    A_hat = D_inv_sqrt @ A_tilde @ D_inv_sqrt

    # ---------- 3) Propagate with early stop ----------
    Y = Y0.copy()
    for it in range(int(n_iter)):
        Y_new = alpha * (A_hat @ Y) + (1.0 - alpha) * Y0
        Y_new = np.asarray(Y_new)
        diff = float(np.mean(np.abs(Y_new - Y)))
        Y = Y_new
        if diff < tol:
            if verbose:
                print(f"[GCN] early stop at iter={it+1}, mean|Δ|={diff:.2e}")
            break

    # ---------- 4) Compute top1/top2/margin ----------
    top1_idx = Y.argmax(axis=1)
    top1 = Y[np.arange(N), top1_idx]

    Y2 = Y.copy()
    Y2[np.arange(N), top1_idx] = -np.inf
    top2 = Y2.max(axis=1)
    margin = top1 - top2

    # ---------- 5) Adaptive gate (key fix vs fixed delta=0.15) ----------
    min_score_eff = float(np.quantile(top1, q_score))
    delta_eff = float(np.quantile(margin, q_margin))

    gate = (top1 >= min_score_eff) & (margin >= delta_eff)

    if verbose:
        print(f"[GCN] adaptive gate: min_score_eff={min_score_eff:.3f}, "
              f"delta_eff={delta_eff:.3f}, pass={gate.mean():.2%}")
        print(f"[GCN] median top1={np.median(top1):.3f}, median margin={np.median(margin):.3f}")

    # preliminary labels + conf
    pred_type = np.array(types, dtype=object)[top1_idx]
    conf = top1.copy()

    # ---------- 6) QUOTA selection to reach target_frac and keep many types ----------
    target_n = int(round(float(target_frac) * N))
    idx_gate = np.where(gate)[0]

    labels = np.array(["unlabeled"] * N, dtype=object)
    seed_conf = np.zeros(N, dtype=float)

    if idx_gate.size == 0 or target_n == 0:
        adata.obs[label_col] = pd.Categorical(labels)
        adata.obs[conf_col] = seed_conf
        return {"Y_propagated": Y, "top1": top1, "top2": top2, "margin": margin, "types": types}

    # group gated indices by predicted type
    gate_df = pd.DataFrame({
        "idx": idx_gate,
        "type": pred_type[idx_gate],
        "conf": conf[idx_gate]
    })

    # ensure each type gets min_cells_per_type if possible
    picked = []
    per_type_min = int(min_cells_per_type)

    for t in types:
        sub = gate_df[gate_df["type"] == t].sort_values("conf", ascending=False)
        if sub.shape[0] == 0:
            continue
        take = min(per_type_min, sub.shape[0])
        picked.extend(sub["idx"].values[:take].tolist())

    picked = list(dict.fromkeys(picked))  # unique preserve order
    remaining_budget = max(0, target_n - len(picked))

    if remaining_budget > 0:
        # fill remaining with highest confidence gated cells not yet picked
        rest = gate_df[~gate_df["idx"].isin(picked)].sort_values("conf", ascending=False)
        picked.extend(rest["idx"].values[:remaining_budget].tolist())

    # assign picked
    labels[picked] = pred_type[picked]
    seed_conf[picked] = conf[picked]

    # ---------- 7) Drop very small types (looser than your pct=0.005) ----------
    min_cells_pct = int(np.floor(float(min_cells_per_type_pct) * N))
    min_cells = max(int(min_cells_floor), int(min_cells_pct), 5)

    vc = pd.Series(labels).value_counts()
    keep_types = set(vc[vc >= min_cells].index.tolist())
    keep_types.discard("unlabeled")

    small_mask = (labels != "unlabeled") & (~pd.Series(labels).isin(keep_types).values)
    labels[small_mask] = "unlabeled"
    seed_conf[small_mask] = 0.0

    # ---------- 8) Write back ----------
    adata.obs[label_col] = pd.Categorical(labels)
    if "unlabeled" not in adata.obs[label_col].cat.categories:
        adata.obs[label_col] = adata.obs[label_col].cat.add_categories(["unlabeled"])
    adata.obs[label_col] = adata.obs[label_col].fillna("unlabeled")
    adata.obs[conf_col] = seed_conf

    if verbose:
        print(f"[GCN] min_cells_per_type used={min_cells}")
        print("[GCN] final label distribution:")
        print(adata.obs[label_col].value_counts())
        print(f"[GCN] labeled fraction={(adata.obs[label_col] != 'unlabeled').mean():.2%}")

    return {"Y_propagated": Y, "top1": top1, "top2": top2, "margin": margin, "types": types}



diag = gcn_seed_labeling_quota(
    adata,
    score_df=marker_score_df,

    # labeling quota
    target_frac=0.6,        # was 0.4 → allow more labeled cells

    # propagation
    temperature=0.1,        # was 0.1 → softer probability distribution
    alpha=0.5,              # was 0.5 → stronger graph propagation
    n_iter=50,              # was 50 → more smoothing

    # graph construction
    neighbors_k=50,          # was 50 → smoother graph
    pca_comps=50,

    label_col="weak_label",
    conf_col="seed_conf",

    verbose=True
)


In [ ]:
#Method2
import numpy as np
import pandas as pd
from sklearn.mixture import BayesianGaussianMixture
from sklearn.preprocessing import StandardScaler

def dp_seed_by_marker_sets_soft(
    adata,
    marker_sets,
    n_components=30,
    weight_concentration_prior=0.05,
    random_state=0,
    per_gene_pos_quantile=0.70,
    cluster_score_min=0.20,
    min_cells_cluster=100,
    label_col_prefix="dp_seed",
    conf_col_prefix="dp_seed_conf",
    verbose=True
):
    diagnostics = {}
    all_vars = set(adata.var_names.astype(str))
    n_cells = adata.n_obs

    # --- Pre-filter thresholds ---
    min_genes_present     = 3
    min_expressed_markers = 2
    min_cells_per_gene    = 20
    min_cell_enrichment   = 0.05

    for setname, genes in marker_sets.items():
        # 0) Keep only markers present in adata
        genes = [g for g in genes if g in all_vars]
        if len(genes) < min_genes_present:
            if verbose:
                print(f"[DP-soft][skip] {setname}: only {len(genes)} markers present in adata.")
            continue

        # 1) Extract expression
        Xg = adata[:, genes].X
        if hasattr(Xg, "toarray"):
            Xg = Xg.toarray()
        Xg = np.asarray(Xg, dtype=float)

        # 2) Pre-filter 1: expressed markers
        expr_per_gene = (Xg > 0).sum(axis=0)
        expressed_markers = int(np.sum(expr_per_gene >= min_cells_per_gene))
        if expressed_markers < min_expressed_markers:
            if verbose:
                print(f"[DP-soft][skip] {setname}: only {expressed_markers} expressed markers.")
            continue

        # 3) Pre-filter 2: cell enrichment
        cells_with_any_marker = (Xg > 0).sum(axis=1) > 0
        cell_enrichment = float(np.mean(cells_with_any_marker))
        if cell_enrichment < min_cell_enrichment:
            if verbose:
                print(f"[DP-soft][skip] {setname}: cell enrichment {cell_enrichment:.4f} < {min_cell_enrichment}.")
            continue

        # 4) Per-gene thresholds
        gene_thr = {}
        for j, g in enumerate(genes):
            v = Xg[:, j]
            pos = v[v > 0]
            if pos.size >= 20:
                gene_thr[g] = float(np.quantile(pos, per_gene_pos_quantile))
            elif pos.size > 0:
                gene_thr[g] = float(np.median(pos))
            else:
                gene_thr[g] = np.inf

        # 5) Cell marker scores
        high_mask = np.zeros_like(Xg, dtype=bool)
        for j, g in enumerate(genes):
            thr = gene_thr[g]
            if np.isinf(thr):
                continue
            high_mask[:, j] = Xg[:, j] > thr

        marker_score = high_mask.sum(axis=1) / max(1, len(genes))

        # 6) Standardize + BGM (NO PCA)
        scaler = StandardScaler(with_mean=True, with_std=True)
        Xg_scaled = scaler.fit_transform(Xg)

        bgm = BayesianGaussianMixture(
            n_components=n_components,
            weight_concentration_prior_type="dirichlet_process",
            weight_concentration_prior=weight_concentration_prior,
            covariance_type="full",
            max_iter=1000,
            n_init=1,
            random_state=random_state,
        )
        bgm.fit(Xg_scaled)
        cl = bgm.predict(Xg_scaled)

        # 7) Cluster-level stats
        df_cluster = pd.DataFrame({"cluster": cl, "score": marker_score})
        cluster_summary = df_cluster.groupby("cluster")["score"].agg(["mean", "count"])

        good_clusters = cluster_summary[
            (cluster_summary["mean"] >= cluster_score_min) &
            (cluster_summary["count"] >= min_cells_cluster)
        ].index.tolist()

        # 8) Assign labels + confidence
        label_col = f"{label_col_prefix}_{setname}"
        conf_col  = f"{conf_col_prefix}_{setname}"

        labels = np.array(["unlabeled"] * n_cells, dtype=object)
        confs  = np.zeros(n_cells, dtype=float)

        if len(good_clusters) > 0:
            max_score = marker_score.max() if marker_score.max() > 0 else 1.0
            for cid in good_clusters:
                idx = np.where(cl == cid)[0]
                labels[idx] = setname
                confs[idx] = marker_score[idx] / max_score

        adata.obs[label_col] = pd.Categorical(labels)
        adata.obs[conf_col]  = confs

        # 9) Diagnostics
        n_lab = (labels != "unlabeled").sum()
        pct_lab = n_lab / n_cells * 100.0
        diagnostics[setname] = {
            "n_markers_used": len(genes),
            "n_clusters": int(len(cluster_summary)),
            "good_clusters": good_clusters,
            "n_labelled": int(n_lab),
            "pct_labelled": float(pct_lab),
            "cluster_summary": cluster_summary,
            "expressed_markers": expressed_markers,
            "cell_enrichment": cell_enrichment,
            "n_features_used": int(Xg_scaled.shape[1]),
        }

        if verbose:
            print(
                f"{setname}: labelled {n_lab}/{n_cells} ({pct_lab:.2f}%), "
                f"good_clusters={len(good_clusters)}, n_features={Xg_scaled.shape[1]}"
            )

    # --- Build consensus weak label across marker sets ---
    label_cols = [f"{label_col_prefix}_{s}" for s in marker_sets.keys() if f"{label_col_prefix}_{s}" in adata.obs]
    conf_cols  = [f"{conf_col_prefix}_{s}" for s in marker_sets.keys() if f"{conf_col_prefix}_{s}" in adata.obs]

    if len(label_cols) > 0:
        lab_df = adata.obs[label_cols].astype(str)
        conf_df = adata.obs[conf_cols].astype(float).fillna(0.0)

        dp_consensus_label = []
        dp_consensus_conf  = []

        for i in adata.obs_names:
            row_labels = lab_df.loc[i].values
            row_confs  = conf_df.loc[i].values
            row_confs = np.where(row_labels == "unlabeled", 0.0, row_confs)

            best_idx = int(np.argmax(row_confs))
            best_conf = float(row_confs[best_idx])
            best_label = row_labels[best_idx] if best_conf > 0 else "unlabeled"

            dp_consensus_label.append(best_label)
            dp_consensus_conf.append(best_conf)

        seed_labels = pd.Categorical(dp_consensus_label)
        if "unlabeled" not in seed_labels.categories:
            seed_labels = seed_labels.add_categories(["unlabeled"])
        adata.obs["weak_label"] = seed_labels
        adata.obs["weak_conf"]  = np.array(dp_consensus_conf, dtype=float)

        # --- Final size filter ---
        #threshold = max(100, int(0.002 * adata.n_obs))
        threshold = max(20, int(0.001 * adata.n_obs))
        counts = adata.obs["weak_label"].value_counts()

        keep_types = set(counts[counts >= threshold].index.tolist())

        mask_keep = adata.obs["weak_label"].isin(keep_types)
        adata.obs.loc[~mask_keep, "weak_label"] = "unlabeled"
        adata.obs.loc[~mask_keep, "weak_conf"]  = 0.0

        if verbose:
            print("\n[DP-soft no PCA] Final weak_label distribution:")
            print(adata.obs["weak_label"].value_counts(normalize=True) * 100)

    return diagnostics

In [ ]:
N = adata.n_obs
#n_components = min(15, int(np.sqrt(N))) # capped adaptive choice
n_components = 15

diag = dp_seed_by_marker_sets_soft(
    adata,
    PBMC_MARKERS,                    # PCA dimension for each marker block
    per_gene_pos_quantile=0.3,
    cluster_score_min=0.08,
    min_cells_cluster=30,
    n_components=n_components,
    weight_concentration_prior=0.1,
    random_state=42
)

for k, v in diag.items():
    print(f"{k}: labelled {v['n_labelled']} cells ({v['pct_labelled']:.2f}%)")

In [ ]:
adata.obs["weak_label"]

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from collections import Counter
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

# -----------------------------
# PCA / neighbors
# -----------------------------
if "X_pca" not in adata.obsm:
    sc.pp.pca(adata, n_comps=20, svd_solver="arpack")

n_pcs_use = min(15, adata.obsm["X_pca"].shape[1])

if "connectivities" not in adata.obsp:
    sc.pp.neighbors(adata, n_neighbors=20, n_pcs=n_pcs_use)
# -----------------------------
# Helper
# -----------------------------
def get_training_data(
    adata,
    X_key="X_pca",
    seed_col="weak_label",
    conf_col="weak_conf",
    min_seed_conf=0.05,
    max_pcs=15
):
    X = np.asarray(adata.obsm[X_key], dtype=np.float32)
    X = X[:, :min(max_pcs, X.shape[1])]
    y = adata.obs[seed_col].astype(str).values

    if conf_col in adata.obs.columns:
        conf = np.asarray(adata.obs[conf_col], dtype=float)
        known_mask = (y != "unlabeled") & (conf >= min_seed_conf)
    else:
        known_mask = (y != "unlabeled")

    y_known = y[known_mask]
    classes = np.unique(y_known)

    print("Known labeled spots:", known_mask.sum())
    print("Known classes:", classes.tolist())

    if known_mask.sum() < 20:
        raise ValueError(f"Too few confident labeled seeds: {known_mask.sum()}")
    if len(classes) < 2:
        raise ValueError(f"Need at least 2 labeled classes, got {len(classes)}: {classes.tolist()}")

    return X, y, known_mask

# -----------------------------
# Methods
# -----------------------------
def rbf_svm_labeling(adata, X_key="X_pca", seed_col="weak_label", conf_col="weak_conf",
                     out_col="label_svm_rbf", out_conf_col="svm_rbf_conf",
                     C=2.0, gamma="scale", min_seed_conf=0.05, min_conf=0.3, max_pcs=15):
    X, y, known_mask = get_training_data(adata, X_key, seed_col, conf_col, min_seed_conf, max_pcs)
    clf = make_pipeline(
        StandardScaler(),
        SVC(C=C, gamma=gamma, kernel="rbf", probability=True, class_weight="balanced", random_state=42)
    )
    clf.fit(X[known_mask], y[known_mask])
    proba = clf.predict_proba(X)
    pred = clf.classes_[np.argmax(proba, axis=1)]
    conf = np.max(proba, axis=1)
    y_out = pred.copy()
    y_out[conf < min_conf] = "unlabeled"
    adata.obs[out_col] = pd.Categorical(y_out)
    adata.obs[out_conf_col] = conf

def kmeans_seed_transfer(adata, X_key="X_pca", seed_col="weak_label", conf_col="weak_conf",
                         out_col="label_kmeans", out_conf_col="kmeans_conf", min_seed_conf=0.05):
    X, y, known_mask = get_training_data(adata, X_key, seed_col, conf_col, min_seed_conf, 30)
    k = min(10, max(8, int(np.sqrt(adata.n_obs / 2))))
    km = KMeans(n_clusters=k, random_state=42, n_init=20, max_iter=500)
    km.fit(X)
    known_types = np.unique(y[known_mask])
    centroids = np.vstack([X[known_mask & (y == t)].mean(axis=0) for t in known_types])

    cluster_to_type, cluster_conf = {}, {}
    for c in np.unique(km.labels_):
        idx = np.where(km.labels_ == c)[0]
        seed_in_c = y[idx]
        seed_in_c = seed_in_c[seed_in_c != "unlabeled"]
        if len(seed_in_c) > 0:
            counts = Counter(seed_in_c)
            maj_label, maj_count = counts.most_common(1)[0]
            cluster_to_type[c] = maj_label
            cluster_conf[c] = maj_count / len(idx)
        else:
            nearest = pairwise_distances(km.cluster_centers_[c].reshape(1, -1), centroids).argmin()
            cluster_to_type[c] = known_types[nearest]
            cluster_conf[c] = 0.0

    labels_out = np.array([cluster_to_type[c] for c in km.labels_], dtype=object)
    conf_out = np.array([cluster_conf[c] for c in km.labels_], dtype=float)
    adata.obs[out_col] = pd.Categorical(labels_out)
    adata.obs[out_conf_col] = conf_out

def knn_labeling(adata, X_key="X_pca", seed_col="weak_label", conf_col="weak_conf",
                 out_col="label_knn", out_conf_col="knn_conf",
                 n_neighbors=9, min_seed_conf=0.05, min_conf=0.3, max_pcs=15):
    X, y, known_mask = get_training_data(adata, X_key, seed_col, conf_col, min_seed_conf, max_pcs)
    k = min(n_neighbors, max(1, known_mask.sum() - 1))
    knn = KNeighborsClassifier(n_neighbors=k, weights="distance", p=2)
    knn.fit(X[known_mask], y[known_mask])
    pred = knn.predict(X)
    proba = knn.predict_proba(X)
    conf = np.max(proba, axis=1)
    y_out = pred.copy()
    y_out[conf < min_conf] = "unlabeled"
    adata.obs[out_col] = pd.Categorical(y_out)
    adata.obs[out_conf_col] = conf

def rf_labeling(adata, X_key="X_pca", seed_col="weak_label", conf_col="weak_conf",
                out_col="label_rf", out_conf_col="rf_conf",
                n_estimators=300, max_depth=18, min_samples_leaf=3,
                min_seed_conf=0.05, min_conf=0.3, max_pcs=15):
    X, y, known_mask = get_training_data(adata, X_key, seed_col, conf_col, min_seed_conf, max_pcs)
    rf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        max_features="sqrt",
        n_jobs=-1,
        class_weight="balanced_subsample",
        random_state=42
    )
    rf.fit(X[known_mask], y[known_mask])
    pred = rf.predict(X)
    proba = rf.predict_proba(X)
    conf = np.max(proba, axis=1)
    y_out = pred.copy()
    y_out[conf < min_conf] = "unlabeled"
    adata.obs[out_col] = pd.Categorical(y_out)
    adata.obs[out_conf_col] = conf

def mlp_labeling(adata, X_key="X_pca", seed_col="weak_label", conf_col="weak_conf",
                 out_col="label_mlp", out_conf_col="mlp_conf",
                 hidden_layer_sizes=(128, 64), alpha=1e-3, max_iter=300,
                 min_seed_conf=0.05, min_conf=0.3, max_pcs=15):
    X, y_str, known_mask = get_training_data(adata, X_key, seed_col, conf_col, min_seed_conf, max_pcs)
    le = LabelEncoder()
    y_known = le.fit_transform(y_str[known_mask])

    clf = make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=hidden_layer_sizes,
            activation="relu",
            solver="adam",
            alpha=alpha,
            learning_rate_init=1e-3,
            max_iter=max_iter,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=20,
            random_state=42
        )
    )
    clf.fit(X[known_mask], y_known)
    proba = clf.predict_proba(X)
    pred_int = np.argmax(proba, axis=1)
    pred_lbl = le.inverse_transform(pred_int)
    conf = np.max(proba, axis=1)
    y_out = pred_lbl.copy()
    y_out[conf < min_conf] = "unlabeled"
    adata.obs[out_col] = pd.Categorical(y_out)
    adata.obs[out_conf_col] = conf

# -----------------------------
# Run methods safely
# -----------------------------
method_outputs = []

for name, func in [
    ("label_svm_rbf", rbf_svm_labeling),
    ("label_kmeans", kmeans_seed_transfer),
    ("label_knn", knn_labeling),
    ("label_rf", rf_labeling),
    ("label_mlp", mlp_labeling),
]:
    try:
        print(f"\nRunning {name} ...")
        func(adata)
        method_outputs.append(name)
        print(f"{name} done")
    except Exception as e:
        print(f"{name} failed: {e}")

print("\nAvailable classifier outputs:", method_outputs)
print("Current obs columns:", adata.obs.columns.tolist())

# -----------------------------
# Consensus from available methods
# -----------------------------
if len(method_outputs) == 0:
    raise ValueError("No classifier output was generated. Check weak_label distribution.")

methods = np.vstack([
    adata.obs[c].astype(str).values
    for c in method_outputs
])

consensus = np.array(["unlabeled"] * adata.n_obs, dtype=object)
agree_frac = np.zeros(adata.n_obs)

for i in range(adata.n_obs):
    votes = [m[i] for m in methods]
    filtered = [v for v in votes if v != "unlabeled"]
    if len(filtered) == 0:
        consensus[i] = "unlabeled"
        agree_frac[i] = 0.0
    else:
        c = Counter(filtered)
        maj_label, maj_votes = c.most_common(1)[0]
        consensus[i] = maj_label
        agree_frac[i] = maj_votes / len(method_outputs)

adata.obs["label_consensus"] = pd.Categorical(consensus)
adata.obs["consensus_agree_frac"] = agree_frac

print("\nConsensus created successfully.")
print(adata.obs["label_consensus"].value_counts(dropna=False))

In [ ]:
adata

In [ ]:
scaicme_labels = adata.obs[
    ["sample", "barcode", "label_consensus"]
].copy()

scaicme_labels.to_csv(
    "scAICME_spatial_lung_labels.csv",
    index=True
)

print(scaicme_labels.head())

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=1.2, key_added="label_leiden")



In [ ]:
print(adata.obs["label_leiden"].value_counts())


# --------------------------------------------------
# Save Leiden labels
# --------------------------------------------------
leiden_output = adata.obs[
    ["sample", "barcode", "label_leiden"]
].copy()

leiden_output.to_csv(
    "Leiden_spatial_labels.csv",
    index=True
)

print("Saved: Leiden_spatial_labels.csv")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

# ============================================================
# GSE292299_SELECTED — NSCLC Lung Cancer
# Original H&E vs scAICME consensus
# Styled like skin spatial plot
# ============================================================

# ------------------------------------------------------------
# Selected NSCLC samples
# ------------------------------------------------------------

sample_info = {
    "P1": "GSM8855703",
    "P2": "GSM8855704",
    "P4": "GSM8855706",
    "P5": "GSM8855707",
}

lung_samples = [
    "P1",
    "P2",
    "P4",
    "P5"
]

# ------------------------------------------------------------
# Choose sample
#
# 0 = P1
# 1 = P2
# 2 = P4
# 3 = P5
# ------------------------------------------------------------

s = 1

sample_name = lung_samples[s]
gsm = sample_info[sample_name]

print(
    "Selected sample:",
    sample_name,
    gsm
)

# ============================================================
# Original individual AnnData
# ============================================================

ad_orig = next(
    x.copy()
    for x in adatas
    if x.obs["sample"].iloc[0] == sample_name
)

ad_plot = ad_orig.copy()

print(
    "Original spots:",
    ad_plot.n_obs
)

# ============================================================
# Obtain H&E image directly from Visium AnnData
# ============================================================

library_id = list(
    ad_orig.uns["spatial"].keys()
)[0]

print(
    "Library ID:",
    library_id
)

spatial_info = (
    ad_orig.uns["spatial"][library_id]
)

# ------------------------------------------------------------
# Hires H&E image
# ------------------------------------------------------------

if (
    "images" in spatial_info
    and
    "hires" in spatial_info["images"]
):

    original_image = np.asarray(
        spatial_info["images"]["hires"]
    )

else:

    raise ValueError(
        f"No hires H&E image found for {sample_name}"
    )

print(
    "Original image shape:",
    original_image.shape
)

# ============================================================
# Hires scale factor
# ============================================================

scale_factor = (
    spatial_info[
        "scalefactors"
    ][
        "tissue_hires_scalef"
    ]
)

print(
    "Hires scale factor:",
    scale_factor
)

# ============================================================
# Transfer scAICME consensus labels from merged lung object
#
# adata should be your scAICME working object:
#
# adata = adata_spatial_lung.copy()
# ============================================================

idx = (
    adata.obs["sample"]
    .astype(str)
    == sample_name
)

merged_sample = (
    adata[idx]
    .copy()
)

print(
    "Merged sample spots:",
    merged_sample.n_obs
)

print(
    "Labeled spots:",
    merged_sample.obs[
        "label_consensus"
    ].notna().sum()
)

# ------------------------------------------------------------
# Check names
# ------------------------------------------------------------

print(
    "\nFirst original obs names:"
)

print(
    ad_plot.obs_names[:5]
)

print(
    "\nFirst merged obs names:"
)

print(
    merged_sample.obs_names[:5]
)

# ============================================================
# Match spots
# ============================================================

common = (
    ad_plot.obs_names
    .intersection(
        merged_sample.obs_names
    )
)

print(
    "\nMatched spot IDs:",
    len(common),
    "/",
    ad_plot.n_obs
)

# initialize
ad_plot.obs[
    "label_consensus"
] = pd.NA

# ------------------------------------------------------------
# Direct transfer using obs_names
# ------------------------------------------------------------

ad_plot.obs.loc[
    common,
    "label_consensus"
] = (
    merged_sample.obs.loc[
        common,
        "label_consensus"
    ]
    .astype(str)
    .values
)

# ============================================================
# Backup barcode matching
# ============================================================

if (
    ad_plot.obs[
        "label_consensus"
    ].notna().sum()
    == 0
):

    print(
        "\nNo obs-name matches."
        "\nTrying barcode matching..."
    )

    label_table = (
        merged_sample.obs[
            [
                "barcode",
                "label_consensus"
            ]
        ]
        .copy()
    )

    label_table[
        "barcode"
    ] = (
        label_table[
            "barcode"
        ].astype(str)
    )

    label_table = (
        label_table
        .drop_duplicates(
            "barcode"
        )
        .set_index(
            "barcode"
        )
    )

    ad_plot.obs[
        "label_consensus"
    ] = (
        ad_plot.obs[
            "barcode"
        ]
        .astype(str)
        .map(
            label_table[
                "label_consensus"
            ]
        )
    )

# ------------------------------------------------------------
# Clean labels
# ------------------------------------------------------------

ad_plot.obs[
    "label_consensus"
] = (
    ad_plot.obs[
        "label_consensus"
    ]
    .astype("string")
    .str.strip()
)

ad_plot.obs.loc[
    ad_plot.obs[
        "label_consensus"
    ].isin(
        [
            "nan",
            "None",
            "<NA>"
        ]
    ),
    "label_consensus"
] = pd.NA

print(
    "\nLabel distribution:"
)

print(
    ad_plot.obs[
        "label_consensus"
    ].value_counts(
        dropna=False
    )
)

# ============================================================
# Keep tissue spots
# ============================================================

if "in_tissue" in ad_orig.obs.columns:

    ad_orig = ad_orig[
        ad_orig.obs[
            "in_tissue"
        ].astype(int)
        == 1
    ].copy()

if "in_tissue" in ad_plot.obs.columns:

    ad_plot = ad_plot[
        ad_plot.obs[
            "in_tissue"
        ].astype(int)
        == 1
    ].copy()

# ------------------------------------------------------------
# Keep only scAICME-labelled spots in right panel
# ------------------------------------------------------------

ad_plot = ad_plot[
    ad_plot.obs[
        "label_consensus"
    ].notna()
].copy()

print(
    "\nFinal plotted labeled spots:",
    ad_plot.n_obs
)

# ============================================================
# HIGH-CONTRAST LUNG-CANCER PALETTE
#
# The exact key must match label_consensus.
# Unrecognised labels get an automatic bright colour below.
# ============================================================

celltype_colors = {

    # --------------------------------------------------------
    # Tumour / epithelial
    # --------------------------------------------------------
    "Tumor_Epithelial": "#E41A1C",       # bright red
    "Epithelial_tumor": "#E41A1C",
    "Malignant_epithelial": "#E41A1C",

    "Alveolar": "#FF7F00",               # orange
    "AT1": "#FF7F00",
    "AT2": "#FDB462",                    # light orange

    # --------------------------------------------------------
    # Stromal
    # --------------------------------------------------------
    "Fibroblast": "#4DAF4A",             # green
    "CAF": "#33A02C",                    # dark green
    "Fibroblast_CAF": "#33A02C",

    "Myofibroblast": "#984EA3",          # purple
    "SmoothMuscle": "#A65628",           # brown
    "Smooth_muscle": "#A65628",

    # --------------------------------------------------------
    # Vascular
    # --------------------------------------------------------
    "Endothelial": "#00BFC4",            # cyan
    "Pericyte": "#1F78B4",               # blue

    # --------------------------------------------------------
    # Lymphoid
    # --------------------------------------------------------
    "T_CELL": "#377EB8",                 # medium blue
    "T_cell": "#377EB8",

    "Cytotoxic_T_NK": "#08306B",         # dark navy
    "NK": "#08306B",
    "NK_cell": "#08306B",

    "B_cell": "#FFD92F",                 # yellow
    "B_CELL": "#FFD92F",

    "Plasma": "#E6AB02",                 # mustard
    "Plasma_cell": "#E6AB02",

    # --------------------------------------------------------
    # Myeloid
    # --------------------------------------------------------
    "Myeloid": "#8C564B",                # brown
    "Macrophage": "#6A3D9A",             # violet
    "Monocyte": "#B15928",                # dark orange-brown
    "Dendritic": "#66C2A5",              # sea green

    # --------------------------------------------------------
    # Other immune
    # --------------------------------------------------------
    "Mast": "#C51B7D",                   # magenta
}

# ============================================================
# Coordinates
# ============================================================

coords_orig = (
    np.asarray(
        ad_orig.obsm[
            "spatial"
        ]
    )
    * scale_factor
)

coords_plot = (
    np.asarray(
        ad_plot.obsm[
            "spatial"
        ]
    )
    * scale_factor
)

labels = (
    ad_plot.obs[
        "label_consensus"
    ]
    .astype(str)
    .values
)

categories = sorted(
    pd.unique(
        labels
    )
)

print(
    "\nCell types found:"
)

print(
    categories
)

# ============================================================
# Automatic high-contrast fallback palette
#
# IMPORTANT:
# Unknown classes are NOT assigned black.
# ============================================================

fallback_colors = [
    "#E41A1C",   # red
    "#377EB8",   # blue
    "#4DAF4A",   # green
    "#984EA3",   # purple
    "#FF7F00",   # orange
    "#00BFC4",   # cyan
    "#F781BF",   # pink
    "#FFD92F",   # yellow
    "#A65628",   # brown
    "#66C2A5",   # sea green
    "#C51B7D",   # magenta
    "#1F78B4",   # deep blue
    "#B15928",   # dark orange
    "#8DA0CB",   # lavender-blue
    "#E78AC3",   # rose
    "#A6D854",   # yellow-green
]

# ------------------------------------------------------------
# Don't reuse a colour already assigned to another category
# when possible
# ------------------------------------------------------------

used_colors = {
    celltype_colors[ct]
    for ct in categories
    if ct in celltype_colors
}

available_colors = [
    c
    for c in fallback_colors
    if c not in used_colors
]

color_index = 0

for ct in categories:

    if ct not in celltype_colors:

        if color_index < len(
            available_colors
        ):

            celltype_colors[
                ct
            ] = available_colors[
                color_index
            ]

            color_index += 1

        else:

            celltype_colors[
                ct
            ] = fallback_colors[
                color_index
                % len(
                    fallback_colors
                )
            ]

            color_index += 1

print(
    "\nFinal colour mapping:"
)

for ct in categories:

    print(
        f"{ct:30s}",
        "->",
        celltype_colors[
            ct
        ]
    )

# ============================================================
# FIGURE
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(16, 7),
    gridspec_kw={
        "wspace": 0.03
    }
)

# ============================================================
# LEFT: Original H&E tissue
# ============================================================

axes[0].imshow(
    original_image
)

# ------------------------------------------------------------
# Optional light spot outlines like skin figure
# ------------------------------------------------------------

axes[0].scatter(
    coords_orig[:, 0],
    coords_orig[:, 1],
    s=18,
    facecolors="none",
    edgecolors="#6E6E6E",
    linewidths=0.5,
    alpha=0.50
)

axes[0].set_title(
    "Original tissue",
    fontsize=13
)

axes[0].axis(
    "off"
)

# ============================================================
# RIGHT: H&E + scAICME consensus
# ============================================================

axes[1].imshow(
    original_image
)

for cell_type in categories:

    mask = (
        labels
        == cell_type
    )

    axes[1].scatter(
        coords_plot[
            mask,
            0
        ],
        coords_plot[
            mask,
            1
        ],

        # same general size as skin/breast plots
        s=28,

        color=(
            celltype_colors[
                cell_type
            ]
        ),

        label=cell_type,

        alpha=0.96,

        edgecolors="none",

        zorder=3
    )

axes[1].set_title(
    "scAICME consensus",
    fontsize=13
)

axes[1].axis(
    "off"
)

# ============================================================
# Force identical image dimensions for both panels
# ============================================================

img_h, img_w = (
    original_image.shape[:2]
)

for ax in axes:

    ax.set_xlim(
        0,
        img_w
    )

    ax.set_ylim(
        img_h,
        0
    )

    ax.set_aspect(
        "equal"
    )

# ============================================================
# Legend
# ============================================================

axes[1].legend(
    bbox_to_anchor=(
        1.02,
        0.5
    ),
    loc="center left",
    frameon=False,
    fontsize=10,
    markerscale=1.2
)

# ============================================================
# Overall title
# ============================================================

fig.suptitle(
    "GSE292299 NSCLC Lung Cancer",
    fontsize=16,
    y=0.98
)

# ============================================================
# Save
# ============================================================

plt.tight_layout(
    rect=[
        0,
        0,
        0.88,
        0.95
    ]
)

outfile = (
    f"GSE292299_"
    f"{sample_name}_"
    f"original_vs_scAICME_consensus.png"
)

plt.savefig(
    outfile,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(
    "\nSaved:",
    outfile
)

In [ ]:
# ============================================================
# Create Lung CAF subset
# ============================================================

caf_types = [

    "Tumor_Epithelial",

    "Fibroblast",

    "Endothelial",

    "Tcell",

    "NK",

    "Bcell",

    "Plasma",

    "Myeloid",

    "TAM_Macrophage",

    "Neutrophil",
]

print("Available labels:")
print(
    adata.obs["label_consensus"]
    .value_counts()
)

adata_caf = adata[
    adata.obs["label_consensus"]
    .astype(str)
    .isin(caf_types)
].copy()

print(
    "\nCAF subset:",
    adata_caf.shape
)

print(
    "\nCAF label distribution:"
)

print(
    adata_caf.obs["label_consensus"]
    .value_counts()
)

In [ ]:
marker_dict_plot = {}

for ct, genes in PBMC_MARKERS.items():

    genes_present = [
        g
        for g in genes
        if g in adata_caf.var_names
    ]

    if len(genes_present) > 0:
        marker_dict_plot[ct] = genes_present

print(marker_dict_plot)

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import pandas as pd

adata_caf.obs["label_consensus"] = pd.Categorical(
    adata_caf.obs["label_consensus"],
    categories=caf_types,
    ordered=True
)

# make matrixplot without internal title
mp = sc.pl.matrixplot(
    adata_caf,
    marker_dict_plot,
    groupby="label_consensus",
    standard_scale="var",
    cmap="RdBu_r",
    dendrogram=False,
    figsize=(18, 6),
    show=False,
    return_fig=True
)

# build figure first
mp.make_figure()

# add title manually with higher position
mp.fig.suptitle(
    "Lung Cancer Marker Gene Expression",
    fontsize=18,
    y=1.02
)

# give a bit more top margin
mp.fig.subplots_adjust(top=0.82)

mp.fig.savefig(
    "GSE292299_Lung_marker_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
print("Number of individual objects:", len(adatas))

for i, x in enumerate(adatas):
    print(
        i,
        x.shape,
        x.obs["sample"].unique()
        if "sample" in x.obs.columns
        else "NO sample column"
    )

print("\nMerged object shape:")
print(adata.shape)

print("\nMerged sample values:")
print(
    adata.obs["sample"]
    .value_counts(dropna=False)
)

In [ ]:
if RUN_COMPARISONS:
    #Comparing metrics
    import numpy as np
    import pandas as pd

    from scipy import sparse
    from sklearn.neighbors import NearestNeighbors
    from sklearn.metrics import silhouette_score
    from sklearn.preprocessing import LabelEncoder


    # ============================================================
    # 1. Build spatial-neighbor graph WITHIN each tissue sample
    # ============================================================

    def build_samplewise_spatial_graph(
        adata,
        sample_col="sample",
        spatial_key="spatial",
        n_neighbors=6
    ):
        """
        Build a symmetric spatial kNN graph separately within each sample.
        Prevents spots from different tissue sections becoming neighbors.
        """

        if spatial_key not in adata.obsm:
            raise ValueError(f"{spatial_key!r} not found in adata.obsm")

        if sample_col not in adata.obs.columns:
            raise ValueError(f"{sample_col!r} not found in adata.obs")

        N = adata.n_obs

        rows = []
        cols = []
        vals = []

        samples = adata.obs[sample_col].astype(str).values

        for s in pd.unique(samples):

            idx = np.where(samples == s)[0]

            if len(idx) < 2:
                continue

            coords = np.asarray(
                adata.obsm[spatial_key][idx],
                dtype=float
            )

            # cannot request more neighbours than cells
            k = min(n_neighbors + 1, len(idx))

            nbrs = NearestNeighbors(
                n_neighbors=k,
                metric="euclidean"
            ).fit(coords)

            neigh = nbrs.kneighbors(
                coords,
                return_distance=False
            )

            for local_i in range(len(idx)):

                global_i = idx[local_i]

                # first neighbour is normally self
                for local_j in neigh[local_i, 1:]:

                    global_j = idx[local_j]

                    rows.append(global_i)
                    cols.append(global_j)
                    vals.append(1.0)

        W = sparse.csr_matrix(
            (vals, (rows, cols)),
            shape=(N, N)
        )

        # make graph symmetric
        W = W.maximum(W.T)

        # remove self edges
        W.setdiag(0)
        W.eliminate_zeros()

        return W

    def spatial_coherence(
        labels,
        W
    ):
        """
        Fraction of spatial edges connecting spots with the same label.

        Higher = better spatial coherence.
        """

        labels = np.asarray(labels).astype(str)

        row, col = W.nonzero()

        if len(row) == 0:
            return np.nan

        same = labels[row] == labels[col]

        coherence = np.mean(same)

        return float(coherence)


    def spatial_inconsistency(
        labels,
        W
    ):
        """
        Fraction of spatial edges connecting different labels.

        Lower = better.
        """

        coherence = spatial_coherence(labels, W)

        return 1.0 - coherence

    def rare_domain_recovery_proxy(
        labels,
        W,
        rare_fraction=0.02,
        min_cells=3,
        exclude_labels=("unlabeled",)
    ):
        """
        Reference-free rare-domain recovery proxy.

        Rare type:
            <= rare_fraction of all labelled spots

        A rare spot is considered spatially recovered if
        at least one spatial neighbour has the same label.

        Higher = better.
        """

        labels = np.asarray(labels).astype(str)

        valid_mask = ~np.isin(labels, exclude_labels)
        valid_labels = labels[valid_mask]

        counts = pd.Series(valid_labels).value_counts()

        N = len(valid_labels)

        rare_types = counts[
            (counts <= rare_fraction * N) &
            (counts >= min_cells)
        ].index.tolist()

        if len(rare_types) == 0:
            return np.nan, {}, []

        per_type = {}

        for t in rare_types:

            idx = np.where(labels == t)[0]

            recovered = 0

            for i in idx:

                neigh = W[i].indices

                if len(neigh) == 0:
                    continue

                if np.any(labels[neigh] == t):
                    recovered += 1

            per_type[t] = recovered / len(idx)

        macro = np.mean(list(per_type.values()))

        return float(macro), per_type, rare_types


    import numpy as np

    # ============================================================
    # Moran's I for numeric vector
    # ============================================================

    def morans_I_numeric(x, W):
        """
        Moran's I for a numeric vector.
        """

        x = np.asarray(x, dtype=float)

        N = len(x)

        x_mean = np.mean(x)
        z = x - x_mean

        S0 = W.sum()

        denominator = np.sum(z ** 2)

        if S0 == 0 or denominator == 0:
            return np.nan

        numerator = z @ (W @ z)

        I = (N / S0) * (numerator / denominator)

        return float(I)


    # ============================================================
    # Moran's I for categorical labels
    # ============================================================

    def categorical_morans_I(
        labels,
        W,
        exclude_labels=("unlabeled",)
    ):
        """
        Calculate one-vs-rest Moran's I for every cell type.

        Example:
            Keratinocyte = 1, all others = 0
            Fibroblast   = 1, all others = 0
            etc.

        Returns
        -------
        macro_I:
            Mean Moran's I across cell types.

        results:
            Dictionary containing Moran's I for each cell type.
        """

        labels = np.asarray(labels).astype(str)

        classes = [
            x for x in np.unique(labels)
            if x not in exclude_labels
        ]

        results = {}

        for cell_type in classes:

            # binary indicator for this cell type
            binary = (labels == cell_type).astype(float)

            I = morans_I_numeric(
                binary,
                W
            )

            results[cell_type] = I

        # remove NaN values before averaging
        valid = [
            v for v in results.values()
            if np.isfinite(v)
        ]

        macro_I = (
            np.mean(valid)
            if len(valid) > 0
            else np.nan
        )

        return float(macro_I), results






In [ ]:
if RUN_COMPARISONS:
    import numpy as np
    import pandas as pd


    def evaluate_spatial_method(
        adata,
        label_col,
        W,
        method_name=None,
        rare_fraction=0.02,
        min_rare_cells=3
    ):
        """
        Evaluate one clustering/annotation method using four spatial metrics.

        Metrics
        -------
        Spatial coherence       : higher is better
        Spatial inconsistency   : lower is better
        Moran's I               : higher is better
        Rare-domain recovery    : higher is better
        """

        if label_col not in adata.obs.columns:
            raise ValueError(
                f"Column '{label_col}' not found in adata.obs"
            )

        labels = adata.obs[label_col].astype(str).values

        # ------------------------------------------
        # 1. Spatial coherence
        # ------------------------------------------
        coherence = spatial_coherence(
            labels,
            W
        )

        # ------------------------------------------
        # 2. Spatial inconsistency
        # ------------------------------------------
        inconsistency = spatial_inconsistency(
            labels,
            W
        )

        # ------------------------------------------
        # 3. Moran's I
        # ------------------------------------------
        moran_macro, moran_per_type = categorical_morans_I(
            labels,
            W
        )

        # ------------------------------------------
        # 4. Rare-domain recovery proxy
        # ------------------------------------------
        rare_recovery, rare_per_type, rare_types = (
            rare_domain_recovery_proxy(
                labels,
                W,
                rare_fraction=rare_fraction,
                min_cells=min_rare_cells
            )
        )

        if method_name is None:
            method_name = label_col

        result = {
            "Method": method_name,
            "Spatial coherence ↑": coherence,
            "Spatial inconsistency ↓": inconsistency,
            "Moran's I ↑": moran_macro,
            "Rare-domain recovery proxy ↑": rare_recovery
        }

        details = {
            "Moran_per_type": moran_per_type,
            "Rare_types": rare_types,
            "Rare_recovery_per_type": rare_per_type
        }

        return result, details

In [ ]:
if RUN_COMPARISONS:
    # ============================================================
    # SpaGCN — GSE292299_SELECTED NSCLC Lung Cancer
    #
    # Run in:
    # conda activate spagcn_env
    # ============================================================

    import SpaGCN as spg
    import scanpy as sc
    import numpy as np
    import pandas as pd
    from scipy import sparse


    # ============================================================
    # 1. Selected NSCLC samples
    # ============================================================

    samples = [
        "P1",
        "P2",
        "P4",
        "P5"
    ]


    # ============================================================
    # 2. Check merged lung object before starting
    # ============================================================

    print("adata shape:", adata.shape)

    print("\nSamples in adata:")
    print(
        adata.obs["sample"]
        .value_counts()
    )

    print(
        "\nTotal spots:",
        adata.n_obs
    )


    # ============================================================
    # 3. Storage for SpaGCN predictions
    # ============================================================

    spagcn_labels = pd.Series(
        index=adata.obs_names.astype(str),
        dtype="object"
    )

    spagcn_summary = []


    # ============================================================
    # 4. Run SpaGCN sample-by-sample
    # ============================================================

    for s in samples:

        print("\n" + "=" * 70)
        print(f"Running SpaGCN: {s}")
        print("=" * 70)

        # --------------------------------------------------------
        # 4.1 Extract one NSCLC tissue section
        # --------------------------------------------------------

        ad = adata[
            adata.obs["sample"]
            .astype(str)
            == s
        ].copy()

        print(
            f"{s}: "
            f"{ad.n_obs} spots × "
            f"{ad.n_vars} genes"
        )

        # safety check
        if ad.n_obs == 0:

            print(
                f"WARNING: {s} contains no spots. Skipping."
            )

            continue


        # ========================================================
        # 5. Use raw counts if available
        # ========================================================

        if "counts" in ad.layers:

            print(
                f"{s}: using counts layer."
            )

            ad.X = (
                ad.layers["counts"]
                .copy()
            )


        # ========================================================
        # 6. Gene filtering
        # ========================================================

        sc.pp.filter_genes(
            ad,
            min_cells=3
        )

        print(
            f"{s}: after gene filtering = "
            f"{ad.n_obs} spots × "
            f"{ad.n_vars} genes"
        )


        # ========================================================
        # 7. Normalize and log-transform
        # ========================================================

        sc.pp.normalize_total(
            ad,
            target_sum=1e4
        )

        sc.pp.log1p(
            ad
        )


        # ========================================================
        # 8. Spatial coordinates
        # ========================================================

        if "spatial" not in ad.obsm:

            print(
                f"WARNING: {s} has no spatial coordinates."
            )

            continue

        coords = np.asarray(
            ad.obsm[
                "spatial"
            ]
        )

        x = coords[:, 0]
        y = coords[:, 1]

        print(
            "Coordinate range:",
            f"x = {x.min():.1f}–{x.max():.1f},",
            f"y = {y.min():.1f}–{y.max():.1f}"
        )


        # ========================================================
        # 9. Spatial adjacency
        #
        # histology=False:
        # use expression + physical spatial coordinates.
        #
        # This matches the previous benchmarking setup.
        # ========================================================

        adj = spg.calculate_adj_matrix(
            x=x,
            y=y,
            histology=False
        )

        print(
            "Adjacency shape:",
            adj.shape
        )


        # ========================================================
        # 10. Search spatial parameter l
        # ========================================================

        l = spg.search_l(
            p=0.5,
            adj=adj,
            start=0.01,
            end=1000,
            tol=0.01,
            max_run=100
        )

        print(
            f"{s}: selected l = {l}"
        )


        # ========================================================
        # 11. Initialize SpaGCN model
        # ========================================================

        clf = spg.SpaGCN()

        clf.set_l(
            l
        )


        # ========================================================
        # 12. Convert expression matrix to dense float32
        #
        # Older SpaGCN versions internally expect dense data.
        # ========================================================

        if sparse.issparse(ad.X):

            ad.X = (
                ad.X
                .toarray()
                .astype(
                    np.float32
                )
            )

        else:

            ad.X = np.asarray(
                ad.X,
                dtype=np.float32
            )

        print(
            f"{s}: expression matrix:",
            ad.X.shape,
            ad.X.dtype
        )


        # ========================================================
        # 13. Train SpaGCN
        # ========================================================

        clf.train(
            ad,
            adj,
            init_spa=True,
            init="louvain",
            res=0.5,
            tol=5e-3,
            lr=0.05,
            max_epochs=200
        )


        # ========================================================
        # 14. Predict spatial domains
        # ========================================================

        pred, prob = clf.predict()

        pred = np.asarray(
            pred
        ).astype(str)


        # ========================================================
        # 15. Store labels
        # ========================================================

        spagcn_labels.loc[
            ad.obs_names.astype(str)
        ] = pred


        # ========================================================
        # 16. Cluster diagnostics
        # ========================================================

        unique_clusters, cluster_counts = np.unique(
            pred,
            return_counts=True
        )

        n_clusters = len(
            unique_clusters
        )

        print(
            f"\n{s}: SpaGCN domains = "
            f"{n_clusters}"
        )

        cluster_distribution = pd.Series(
            cluster_counts,
            index=unique_clusters
        ).sort_index()

        print(
            cluster_distribution
        )


        # ========================================================
        # 17. Store diagnostic information
        # ========================================================

        spagcn_summary.append({

            "sample":
                s,

            "n_spots":
                ad.n_obs,

            "n_genes":
                ad.n_vars,

            "n_clusters":
                n_clusters,

            "l":
                l
        })


    # ============================================================
    # 18. Add SpaGCN labels to full NSCLC AnnData
    # ============================================================

    adata.obs[
        "label_SpaGCN"
    ] = pd.Categorical(
        spagcn_labels.reindex(
            adata.obs_names.astype(str)
        )
    )


    # ============================================================
    # 19. Final results
    # ============================================================

    print("\n" + "=" * 70)

    print(
        "FINAL SpaGCN RESULTS — "
        "GSE292299 NSCLC"
    )

    print("=" * 70)


    n_labeled = (
        adata.obs[
            "label_SpaGCN"
        ]
        .notna()
        .sum()
    )

    print(
        "\nTotal labeled spots:",
        n_labeled,
        "/",
        adata.n_obs
    )

    print(
        "Percentage labeled:",
        round(
            100
            * n_labeled
            / adata.n_obs,
            2
        ),
        "%"
    )


    # ============================================================
    # 20. Overall SpaGCN label distribution
    # ============================================================

    print(
        "\nOverall SpaGCN label distribution:"
    )

    print(
        adata.obs[
            "label_SpaGCN"
        ]
        .value_counts(
            dropna=False
        )
    )


    # ============================================================
    # 21. Sample-wise cluster distribution
    # ============================================================

    print(
        "\nSample-wise SpaGCN clusters:"
    )

    print(
        pd.crosstab(
            adata.obs[
                "sample"
            ],
            adata.obs[
                "label_SpaGCN"
            ]
        )
    )


    # ============================================================
    # 22. Diagnostic summary
    # ============================================================

    spagcn_summary_df = pd.DataFrame(
        spagcn_summary
    )

    print(
        "\nSpaGCN summary:"
    )

    print(
        spagcn_summary_df.to_string(
            index=False
        )
    )


    # ============================================================
    # 23. Save SpaGCN labels
    # ============================================================

    spagcn_output = (
        adata.obs[
            [
                "sample",
                "barcode",
                "label_SpaGCN"
            ]
        ]
        .copy()
    )

    spagcn_output.to_csv(
        "GSE292299_SpaGCN_spatial_labels.csv",
        index=True
    )


    # ============================================================
    # 24. Save diagnostic summary
    # ============================================================

    spagcn_summary_df.to_csv(
        "GSE292299_SpaGCN_summary.csv",
        index=False
    )


    # ============================================================
    # 25. Finished
    # ============================================================

    print("\nSaved:")

    print(
        "GSE292299_SpaGCN_spatial_labels.csv"
    )

    print(
        "GSE292299_SpaGCN_summary.csv"
    )

In [ ]:
if RUN_COMPARISONS:
    # ============================================================
    # MEMORY-SAFE stLearn
    # GSE292299_SELECTED — NSCLC Lung Cancer
    #
    # Run in:
    # conda activate stlearn_env
    # ============================================================

    import os
    import gc
    import stlearn as st
    import scanpy as sc
    import pandas as pd
    import numpy as np


    # ============================================================
    # 1. Selected lung-cancer samples
    # ============================================================

    samples = [
        "P1",
        "P2",
        "P4",
        "P5"
    ]


    # ============================================================
    # 2. Parameters
    # ============================================================

    N_HVG = 2500
    N_CLUSTERS = 8
    N_PCS = 30


    # ============================================================
    # 3. Check merged lung object
    # ============================================================

    print("adata shape:", adata.shape)

    print("\nSamples in adata:")
    print(
        adata.obs["sample"]
        .value_counts()
    )

    print(
        "\nTotal spots:",
        adata.n_obs
    )


    # ============================================================
    # 4. Output storage
    # ============================================================

    stlearn_labels = pd.Series(
        index=adata.obs_names.astype(str),
        dtype="object"
    )

    stlearn_summary = []


    # ============================================================
    # 5. Run stLearn independently for each tissue section
    # ============================================================

    for s in samples:

        print("\n" + "=" * 72)
        print(f"Running memory-safe stLearn: {s}")
        print("=" * 72)

        # --------------------------------------------------------
        # Clean memory before starting new sample
        # --------------------------------------------------------

        gc.collect()

        # ========================================================
        # 5.1 Get individual Visium object
        # ========================================================

        matching_objects = [
            x
            for x in adatas
            if (
                "sample" in x.obs.columns
                and
                x.obs["sample"].iloc[0] == s
            )
        ]

        if len(matching_objects) == 0:

            print(
                f"WARNING: no individual AnnData found for {s}."
            )

            continue

        ad_st = matching_objects[0].copy()

        print(
            f"{s}: {ad_st.n_obs} spots × "
            f"{ad_st.n_vars} genes"
        )

        if ad_st.n_obs == 0:

            print(
                f"WARNING: {s} has no spots. Skipping."
            )

            del ad_st
            gc.collect()

            continue


        # ========================================================
        # 5.2 Check spatial information
        # ========================================================

        if "spatial" not in ad_st.obsm:

            print(
                f"WARNING: spatial coordinates missing for {s}."
            )

            del ad_st
            gc.collect()

            continue

        print(
            f"{s}: spatial coordinates =",
            ad_st.obsm["spatial"].shape
        )

        print(
            f"{s}: spatial libraries =",
            list(
                ad_st.uns.get(
                    "spatial",
                    {}
                ).keys()
            )
        )


        # ========================================================
        # 5.3 Convert Scanpy Visium object to stLearn
        # ========================================================

        ad_st = st.convert_scanpy(
            ad_st,
            use_quality="hires"
        )

        print(
            f"{s}: converted to stLearn format."
        )


        # ========================================================
        # 6. Gene filtering
        # ========================================================

        st.pp.filter_genes(
            ad_st,
            min_cells=3
        )

        print(
            f"{s}: after filter_genes = "
            f"{ad_st.n_obs} × {ad_st.n_vars}"
        )


        # ========================================================
        # 7. Normalize
        # ========================================================

        st.pp.normalize_total(
            ad_st,
            target_sum=1e4
        )


        # ========================================================
        # 8. Log transformation
        # ========================================================

        st.pp.log1p(
            ad_st
        )

        print(
            f"{s}: before HVG selection = "
            f"{ad_st.n_obs} × {ad_st.n_vars}"
        )


        # ========================================================
        # 9. Select HVGs
        #
        # CRITICAL MEMORY-SAVING STEP
        # ========================================================

        n_hvg_use = min(
            N_HVG,
            ad_st.n_vars
        )

        sc.pp.highly_variable_genes(
            ad_st,
            n_top_genes=n_hvg_use,
            flavor="seurat"
        )

        n_hvg_found = int(
            ad_st.var[
                "highly_variable"
            ].sum()
        )

        print(
            f"{s}: HVGs detected =",
            n_hvg_found
        )

        # --------------------------------------------------------
        # Retain only HVGs
        # --------------------------------------------------------

        ad_st = ad_st[
            :,
            ad_st.var[
                "highly_variable"
            ].values
        ].copy()

        print(
            f"{s}: after HVG selection = "
            f"{ad_st.n_obs} × {ad_st.n_vars}"
        )


        # ========================================================
        # 10. Convert X to float32 when dense
        # ========================================================

        if not hasattr(
            ad_st.X,
            "tocsr"
        ):

            ad_st.X = np.asarray(
                ad_st.X,
                dtype=np.float32
            )


        # ========================================================
        # 11. Store reduced expression in .raw
        #
        # This now contains only ~2500 genes, not 18,000+
        # ========================================================

        ad_st.raw = ad_st.copy()


        # ========================================================
        # 12. Initial PCA
        # ========================================================

        n_pcs = min(
            N_PCS,
            ad_st.n_obs - 1,
            ad_st.n_vars - 1
        )

        st.em.run_pca(
            ad_st,
            n_comps=n_pcs
        )

        print(
            f"{s}: initial X_pca =",
            ad_st.obsm[
                "X_pca"
            ].shape
        )


        # ========================================================
        # 13. H&E tiling
        # ========================================================

        tile_path = os.path.join(
            "stlearn_tiles_GSE292299",
            s
        )

        os.makedirs(
            tile_path,
            exist_ok=True
        )

        st.pp.tiling(
            ad_st,
            out_path=tile_path
        )

        print(
            f"{s}: image tiles generated."
        )


        # ========================================================
        # 14. Extract morphology features
        # ========================================================

        st.pp.extract_feature(
            ad_st
        )

        print(
            f"{s}: X_morphology =",
            ad_st.obsm[
                "X_morphology"
            ].shape
        )


        # ========================================================
        # 15. SME normalization
        #
        # Uses:
        # physical distance
        # gene expression
        # morphology
        #
        # Now only ~2500 genes enter SME.
        # ========================================================

        print(
            f"\n{s}: starting SME normalization..."
        )

        ad_sme = ad_st.copy()

        st.spatial.sme.sme_normalize(
            ad_sme,
            use_data="raw",
            weights="weights_matrix_all"
        )

        print(
            f"{s}: SME normalized =",
            ad_sme.obsm[
                "raw_SME_normalized"
            ].shape
        )


        # ========================================================
        # 16. Replace X with SME-normalized expression
        # ========================================================

        ad_sme.X = np.asarray(
            ad_sme.obsm[
                "raw_SME_normalized"
            ],
            dtype=np.float32
        )


        # ========================================================
        # 17. Free large temporary SME matrices
        # ========================================================

        if "imputed_data" in ad_sme.obsm:

            del ad_sme.obsm[
                "imputed_data"
            ]

        # We already copied this into X
        if "raw_SME_normalized" in ad_sme.obsm:

            del ad_sme.obsm[
                "raw_SME_normalized"
            ]

        gc.collect()


        # ========================================================
        # 18. Scale SME matrix
        # ========================================================

        st.pp.scale(
            ad_sme
        )

        # make sure dense matrix stays float32
        ad_sme.X = np.asarray(
            ad_sme.X,
            dtype=np.float32
        )


        # ========================================================
        # 19. PCA after SME
        # ========================================================

        n_pcs_sme = min(
            N_PCS,
            ad_sme.n_obs - 1,
            ad_sme.n_vars - 1
        )

        st.em.run_pca(
            ad_sme,
            n_comps=n_pcs_sme
        )

        print(
            f"{s}: post-SME X_pca =",
            ad_sme.obsm[
                "X_pca"
            ].shape
        )


        # ========================================================
        # 20. K-means spatial-domain clustering
        # ========================================================

        st.tl.clustering.kmeans(
            ad_sme,
            n_clusters=N_CLUSTERS,
            use_data="X_pca",
            key_added="stLearn_cluster"
        )


        # ========================================================
        # 21. Cluster distribution
        # ========================================================

        print(
            f"\n{s} cluster distribution:"
        )

        cluster_distribution = (
            ad_sme.obs[
                "stLearn_cluster"
            ]
            .value_counts()
            .sort_index()
        )

        print(
            cluster_distribution
        )


        # ========================================================
        # 22. Extract labels
        # ========================================================

        labels = (
            ad_sme.obs[
                "stLearn_cluster"
            ]
            .astype(str)
            .copy()
        )


        # ========================================================
        # 23. Transfer labels to merged object
        #
        # obs_names are already:
        #
        # P1_AACA...
        # P2_AACA...
        # P4_AACA...
        # P5_AACA...
        # ========================================================

        common = (
            labels.index
            .intersection(
                stlearn_labels.index
            )
        )

        print(
            f"{s}: matched "
            f"{len(common)}/{len(labels)} spots"
        )

        stlearn_labels.loc[
            common
        ] = labels.loc[
            common
        ].values


        # ========================================================
        # 24. Save diagnostics
        # ========================================================

        stlearn_summary.append(
            {
                "sample":
                    s,

                "n_spots":
                    ad_sme.n_obs,

                "n_genes_HVG":
                    ad_sme.n_vars,

                "n_HVG_requested":
                    N_HVG,

                "n_clusters":
                    ad_sme.obs[
                        "stLearn_cluster"
                    ].nunique(),

                "n_matched":
                    len(common),

                "match_percent":
                    (
                        100.0
                        * len(common)
                        / len(labels)
                    )
            }
        )


        # ========================================================
        # 25. MEMORY CLEANUP AFTER EACH SAMPLE
        # ========================================================

        del labels
        del cluster_distribution
        del ad_sme
        del ad_st

        gc.collect()

        print(
            f"\n{s}: memory cleanup complete."
        )


    # ============================================================
    # 26. Add final stLearn labels to merged lung object
    # ============================================================

    adata.obs[
        "label_stLearn"
    ] = pd.Categorical(
        stlearn_labels.reindex(
            adata.obs_names.astype(str)
        )
    )


    # ============================================================
    # 27. Final results
    # ============================================================

    print("\n" + "=" * 72)

    print(
        "FINAL MEMORY-SAFE stLearn RESULTS — "
        "GSE292299 NSCLC"
    )

    print("=" * 72)


    n_labeled = (
        adata.obs[
            "label_stLearn"
        ]
        .notna()
        .sum()
    )

    print(
        "\nLabeled spots:",
        n_labeled,
        "/",
        adata.n_obs
    )

    print(
        "Percentage labeled:",
        round(
            100
            * n_labeled
            / adata.n_obs,
            2
        ),
        "%"
    )


    # ============================================================
    # 28. Overall cluster distribution
    # ============================================================

    print(
        "\nOverall stLearn cluster distribution:"
    )

    print(
        adata.obs[
            "label_stLearn"
        ]
        .value_counts(
            dropna=False
        )
    )


    # ============================================================
    # 29. Sample-wise cluster distribution
    # ============================================================

    print(
        "\nSample-wise stLearn clusters:"
    )

    print(
        pd.crosstab(
            adata.obs[
                "sample"
            ],
            adata.obs[
                "label_stLearn"
            ]
        )
    )


    # ============================================================
    # 30. Summary dataframe
    # ============================================================

    stlearn_summary_df = pd.DataFrame(
        stlearn_summary
    )

    print(
        "\nstLearn summary:"
    )

    print(
        stlearn_summary_df.to_string(
            index=False
        )
    )


    # ============================================================
    # 31. Save labels
    # ============================================================

    stlearn_output = (
        adata.obs[
            [
                "sample",
                "barcode",
                "label_stLearn"
            ]
        ]
        .copy()
    )

    stlearn_output.to_csv(
        "GSE292299_stLearn_spatial_labels.csv",
        index=True
    )


    # ============================================================
    # 32. Save summary
    # ============================================================

    stlearn_summary_df.to_csv(
        "GSE292299_stLearn_summary.csv",
        index=False
    )


    # ============================================================
    # 33. Finished
    # ============================================================

    print("\nSaved:")

    print(
        "GSE292299_stLearn_spatial_labels.csv"
    )

    print(
        "GSE292299_stLearn_summary.csv"
    )

In [ ]:
if RUN_COMPARISONS:
    # ============================================================
    # FINAL SPATIAL BENCHMARK
    # scAICME vs Leiden vs SpaGCN vs stLearn
    # ============================================================

    import numpy as np
    import pandas as pd

    from scipy import sparse
    from sklearn.neighbors import NearestNeighbors


    # ============================================================
    # 1. LOAD SAVED LABEL FILES
    # ============================================================

    scaicme_df = pd.read_csv("scAICME_spatial_lung_labels.csv")
    leiden_df  = pd.read_csv("Leiden_spatial_labels.csv")
    spagcn_df  = pd.read_csv("GSE292299_SpaGCN_spatial_labels.csv")
    stlearn_df = pd.read_csv("GSE292299_stLearn_spatial_labels.csv")


    print("scAICME:", scaicme_df.shape)
    print("Leiden :", leiden_df.shape)
    print("SpaGCN :", spagcn_df.shape)
    print("stLearn:", stlearn_df.shape)

In [ ]:
if RUN_COMPARISONS:
    # --------------------------------------------------
    # Create spot_key in all four dataframes
    # --------------------------------------------------

    for df in [
        scaicme_df,
        leiden_df,
        spagcn_df,
        stlearn_df
    ]:
        df["sample"] = df["sample"].astype(str)
        df["barcode"] = df["barcode"].astype(str)

        df["spot_key"] = (
            df["sample"]
            + "::"
            + df["barcode"]
        )

In [ ]:
if RUN_COMPARISONS:
    comparison_labels = (
        scaicme_df[
            [
                "sample",
                "barcode",
                "spot_key",
                "label_consensus"
            ]
        ]
        .merge(
            leiden_df[
                [
                    "spot_key",
                    "label_leiden"
                ]
            ],
            on="spot_key",
            how="inner"
        )
        .merge(
            spagcn_df[
                [
                    "spot_key",
                    "label_SpaGCN"
                ]
            ],
            on="spot_key",
            how="inner"
        )
        .merge(
            stlearn_df[
                [
                    "spot_key",
                    "label_stLearn"
                ]
            ],
            on="spot_key",
            how="inner"
        )
    )

In [ ]:
if RUN_COMPARISONS:
    print(comparison_labels.shape)

    print(
        comparison_labels[
            [
                "label_consensus",
                "label_leiden",
                "label_SpaGCN",
                "label_stLearn"
            ]
        ].notna().sum()
    )

In [ ]:
if RUN_COMPARISONS:
    adata.obs["spot_key"] = (
        adata.obs["sample"].astype(str)
        + "::"
        + adata.obs["barcode"].astype(str)
    )

    keep = (
        adata.obs["spot_key"]
        .isin(comparison_labels["spot_key"])
    )

    adata_compare = adata[
        keep
    ].copy()

    print(adata_compare)

In [ ]:
if RUN_COMPARISONS:
    lab = comparison_labels.set_index(
        "spot_key"
    )

In [ ]:
if RUN_COMPARISONS:
    for col in [
        "label_consensus",
        "label_leiden",
        "label_SpaGCN",
        "label_stLearn"
    ]:

        adata_compare.obs[col] = (
            adata_compare.obs["spot_key"]
            .map(lab[col])
            .astype("category")
        )

In [ ]:
if RUN_COMPARISONS:
    for col in [
        "label_consensus",
        "label_leiden",
        "label_SpaGCN",
        "label_stLearn"
    ]:

        print("\n", col)

        print(
            adata_compare.obs[col]
            .value_counts(dropna=False)
        )

In [ ]:
if RUN_COMPARISONS:
    print(
        adata_compare.obs[
            [
                "label_consensus",
                "label_leiden",
                "label_SpaGCN",
                "label_stLearn"
            ]
        ].isna().sum()
    )

In [ ]:
if RUN_COMPARISONS:
    W_compare = build_samplewise_spatial_graph(
        adata_compare,
        sample_col="sample",
        spatial_key="spatial",
        n_neighbors=6
    )

    print(W_compare.shape)

In [ ]:
if RUN_COMPARISONS:
    methods = {
        "scAICME": "label_consensus",
        "Leiden": "label_leiden",
        "SpaGCN": "label_SpaGCN",
        "stLearn": "label_stLearn"
    }

    comparison_results = []
    comparison_details = {}

    for method_name, label_col in methods.items():

        result, details = evaluate_spatial_method(
            adata=adata_compare,
            label_col=label_col,
            W=W_compare,
            method_name=method_name,
            rare_fraction=0.02,
            min_rare_cells=3
        )

        comparison_results.append(result)
        comparison_details[method_name] = details


    final_results = pd.DataFrame(
        comparison_results
    )

    final_results = final_results[
        [
            "Method",
            "Spatial coherence ↑",
            "Spatial inconsistency ↓",
            "Moran's I ↑",
        ]
    ]

    final_results

In [ ]:
if RUN_COMPARISONS:
    final_results.to_csv(
        "Spatial_methods_final_lung_results.csv",
        index=False
    )